# IRA-900-C continuous-weather process bridge (v7)

## What is fixed in this version

This notebook removes three shortcuts from earlier exploratory versions:

1. **No ambient RH threshold.**
   Weather is never converted into "adsorb below X% / desorb above Y%" states.

2. **No independent hourly cycles.**
   An hourly weather point is not treated as a complete adsorption→regeneration cycle.

3. **No 400-ppm equilibrium state as regeneration.**
   Wang regeneration uses humid N2 / low CO2 partial pressure. The 400-ppm q(RH,T) surface is used only for the ambient adsorption equilibrium target.

The physical sequence is:

\[
T(t),RH(t)
\rightarrow q^*_{\rm ads}(t)
\rightarrow q(t)
\rightarrow \text{explicit regeneration event}
\rightarrow q(t^+)
\]

The ambient forcing remains continuous between regeneration events.


## What can be answered now

The manuscript already defines

\[
H=\frac12\sum |q^*_{t+1}-q^*_t|.
\]

Here we build the process bridge without inventing an RH switching rule.

Two ambient state models are provided:

- **EQ trajectory**: \(q(t)=q^*(T(t),RH(t))\), useful as the fast-response limit.
- **LDF trajectory**: continuous state propagation
  \[
  \frac{dq}{dt}=k[q^*(t)-q(t)]
  \]
  with user-declared \(\tau_k=1/k\).

Regeneration is a separate scheduled event. The default schedule is deliberately disabled until explicitly chosen.

Wang source constraints stored here:
- adsorption measurements: 400 ppm CO2;
- humid-N2 regeneration at 25 °C;
- 20% RH N2 purge removes ~0.39 mmol/g;
- increasing purge to 50% RH removes an additional ~0.88 mmol/g;
- increasing to 87.5% RH removes an additional ~0.41 mmol/g;
- 50% RH with timely CO2 removal gives a reported theoretical maximum working capacity of 1.27 mmol/g.

These are process regeneration data, not ambient weather thresholds.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np, pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.stats import kendalltau, spearmanr

ROOT=Path('/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/process_bridge')
COMMON=ROOT/'inputs_common'; IRA=ROOT/'branch_ira'; RESULTS=ROOT/'results'; IRA_RESULTS=IRA/'results'
WEATHER=COMMON/'climate_hourly_2015_2024.csv'
PROCESS_MASTER=IRA/'inputs_raw'/'process_dataset_9_materials_exhaustive_audit_SOURCE.xlsx'
H_BASE=RESULTS/'v3_H_reduced_full_matched.csv'

for p in [ROOT,COMMON,IRA,RESULTS,IRA_RESULTS,WEATHER,PROCESS_MASTER,H_BASE]:
    assert p.exists(), f'Missing canonical input: {p}'

RECORD_SCALAR='scalar_observation'
MATERIAL='material'; DTYPE='data_type'; VAR='variable'; TCOL='T_C'; RHCOL='RH_pct'; VCOL='value'
WT='temperature_C'; WRH='RH_pct'; WCITY='city'; WTIME='timestamp_utc'
IRA_MATERIAL='IRA-900-C'

# Hard prohibition: ambient weather thresholds are not parameters in this notebook.
AMBIENT_RH_ADS_THRESHOLD=None
AMBIENT_RH_DES_THRESHOLD=None
assert AMBIENT_RH_ADS_THRESHOLD is None and AMBIENT_RH_DES_THRESHOLD is None
print('Canonical paths verified. Ambient RH thresholds: DISABLED.')


In [ ]:
weather=pd.read_csv(WEATHER)
weather[WTIME]=pd.to_datetime(weather[WTIME],utc=True)
structured=pd.read_excel(PROCESS_MASTER,sheet_name='structured_data')
ira=structured[structured[MATERIAL]==IRA_MATERIAL].copy()
co2=ira[(ira.record_type==RECORD_SCALAR)&(ira[DTYPE]=='equilibrium_CO2')&(ira[VAR]=='q_CO2')].copy()
for c in [TCOL,RHCOL,VCOL]: co2[c]=pd.to_numeric(co2[c],errors='coerce')
co2=co2.dropna(subset=[TCOL,RHCOL,VCOL])
assert set(np.round(co2[TCOL].unique(),6))=={15.,25.,35.}
print('IRA equilibrium rows:',len(co2))
display(co2.sort_values([TCOL,RHCOL]))


In [ ]:
def slice_xy(g,T):
    z=(g[np.isclose(g[TCOL],T)][[RHCOL,VCOL]].dropna()
       .groupby(RHCOL,as_index=False)[VCOL].mean().sort_values(RHCOL))
    return z[RHCOL].to_numpy(float),z[VCOL].to_numpy(float)

def q_slice(g,T,RH):
    RH=np.asarray(RH,float); x,y=slice_xy(g,T); out=np.full(len(RH),np.nan)
    if len(x)<2:return out
    m=(RH>=x.min())&(RH<=x.max())
    out[m]=PchipInterpolator(x,y,extrapolate=False)(RH[m])
    return out

def q_reduced(g,T,RH,Tref=25.,Tmin=15.,Tmax=35.):
    T=np.asarray(T,float); RH=np.asarray(RH,float)
    out=q_slice(g,Tref,RH); out[(T<Tmin)|(T>Tmax)]=np.nan
    return out

def q_full(g,T,RH):
    T=np.asarray(T,float); RH=np.asarray(RH,float)
    Ts=np.array(sorted(g[TCOL].unique()),float)
    qT=np.vstack([q_slice(g,t,RH) for t in Ts]); out=np.full(len(T),np.nan)
    for j in range(len(Ts)-1):
        t0,t1=Ts[j],Ts[j+1]
        m=(T>=t0)&(T<=t1)&np.isfinite(qT[j])&np.isfinite(qT[j+1])
        if m.any():
            w=(T[m]-t0)/(t1-t0); out[m]=(1-w)*qT[j,m]+w*qT[j+1,m]
    return out


## Continuous ambient trajectory

Every hourly \(T,RH\) value is evaluated continuously on the measured equilibrium surface.

No `RH < ...`, `RH > ...`, `np.where(RH...)`, or two-state weather rule is used.

Unsupported states remain NaN. They are not silently extrapolated.


In [ ]:
ws=weather.sort_values([WCITY,WTIME]).reset_index(drop=True)
T=ws[WT].to_numpy(float); RH=ws[WRH].to_numpy(float)
ws['qstar_R']=q_reduced(co2,T,RH)
ws['qstar_F']=q_full(co2,T,RH)
ws['common_supported']=np.isfinite(ws.qstar_R)&np.isfinite(ws.qstar_F)

print('cities:',ws[WCITY].nunique())
print('common supported fraction:',ws.common_supported.mean())


## Continuous-state propagation

The LDF model below propagates sorbent state hour by hour inside contiguous supported segments.

It does **not** reset because RH crosses a threshold.

For unsupported gaps, two transparent options are provided:

- `hold`: retain the last sorbent loading through the gap;
- `nan`: stop process inference across the gap.

`hold` is a bounded memory assumption, not a claim that the sorbent is at equilibrium outside measured support.


In [ ]:
def propagate_city(g,qstar_col,tau_h=1.0,gap_mode='hold'):
    qstar=g[qstar_col].to_numpy(float)
    q=np.full(len(g),np.nan)
    if len(g)==0:return q
    first=np.flatnonzero(np.isfinite(qstar))
    if len(first)==0:return q
    i0=first[0]; q[i0]=qstar[i0]
    alpha=1-np.exp(-1.0/tau_h) if tau_h>0 else 1.0
    for i in range(i0+1,len(g)):
        if np.isfinite(qstar[i]):
            prev=q[i-1]
            if not np.isfinite(prev): prev=qstar[i-1] if np.isfinite(qstar[i-1]) else qstar[i]
            q[i]=prev+alpha*(qstar[i]-prev)
        elif gap_mode=='hold' and np.isfinite(q[i-1]):
            q[i]=q[i-1]
    return q

TAU_HOURS=1.0
GAP_MODE='hold'

parts=[]
for city,g in ws.groupby(WCITY,sort=False):
    z=g.copy()
    z['q_R']=propagate_city(z,'qstar_R',TAU_HOURS,GAP_MODE)
    z['q_F']=propagate_city(z,'qstar_F',TAU_HOURS,GAP_MODE)
    parts.append(z)
traj=pd.concat(parts,ignore_index=True)
print('Continuous LDF trajectories built; tau_h=',TAU_HOURS,'gap_mode=',GAP_MODE)


## Regeneration is an explicit event, not a weather threshold

Wang's regeneration experiments are encoded as process actions.

The source-reported cumulative CO₂ removal from a bed initially saturated near 20% RH, 25 °C is:

- switch 400 ppm CO₂ → N₂ at ~20% RH: 0.39 mmol/g;
- then 50% RH N₂: +0.88 mmol/g;
- then 87.5% RH N₂: +0.41 mmol/g.

Thus the 50% RH process benchmark corresponds to 1.27 mmol/g cumulative removal.

These values define what a **regeneration event can remove under that protocol**. They do not decide when ambient weather triggers regeneration.


In [ ]:
WANG_REGEN=pd.DataFrame([
    {'protocol':'N2_20RH','T_regen_C':25.0,'RH_regen_pct':20.0,'cumulative_CO2_removed_mmol_g':0.39},
    {'protocol':'N2_50RH','T_regen_C':25.0,'RH_regen_pct':50.0,'cumulative_CO2_removed_mmol_g':1.27},
    {'protocol':'N2_87p5RH','T_regen_C':25.0,'RH_regen_pct':87.5,'cumulative_CO2_removed_mmol_g':1.68},
])
display(WANG_REGEN)


## Cycle-control rule

A process cannot be calculated until we say **when the device regenerates**.

This is not an RH threshold.

Examples of legitimate control rules include:
- fixed calendar interval;
- fixed operating duration;
- externally scheduled regeneration;
- loading-based control, if independently justified.

The default below is `None`, so the notebook will not silently invent a cycle frequency.

For a first sensitivity test, set `REGEN_INTERVAL_HOURS` explicitly (e.g. 24, 48, 168). This means "regenerate every N operating hours", not "regenerate when RH reaches X".


In [ ]:
REGEN_INTERVAL_HOURS=None
REGEN_PROTOCOL='N2_50RH'

def scheduled_regen_indices(n,interval_h):
    if interval_h is None:return np.array([],dtype=int)
    assert interval_h>=1
    return np.arange(interval_h-1,n,interval_h,dtype=int)

if REGEN_INTERVAL_HOURS is None:
    print('PROCESS SIMULATION GATED: choose an explicit non-RH regeneration schedule.')
else:
    print('Regeneration every',REGEN_INTERVAL_HOURS,'h using',REGEN_PROTOCOL)


In [ ]:
def run_scheduled_process(g,q_col,interval_h,regen_capacity):
    # Continuous ambient loading trajectory is preserved.
    # At each scheduled event, recover at most the CO2 currently represented
    # above the post-regeneration state implied by the source removal capacity.
    # This is a bounded recovery accounting model, not a kinetic regeneration model.
    q=g[q_col].to_numpy(float).copy()
    idx=scheduled_regen_indices(len(g),interval_h)
    recovered=[]
    last_event=-1
    for i in idx:
        if not np.isfinite(q[i]): continue
        # Source experiment constrains maximum removable amount per event.
        rec=min(max(q[i],0.0),regen_capacity)
        recovered.append((i,rec))
    return recovered

process_rows=[]
if REGEN_INTERVAL_HOURS is not None:
    cap=float(WANG_REGEN.loc[WANG_REGEN.protocol==REGEN_PROTOCOL,'cumulative_CO2_removed_mmol_g'].iloc[0])
    for city,g in traj.groupby(WCITY,sort=False):
        for rep,qcol in [('reduced','q_R'),('full','q_F')]:
            ev=run_scheduled_process(g.reset_index(drop=True),qcol,REGEN_INTERVAL_HOURS,cap)
            vals=np.array([x[1] for x in ev],float)
            process_rows.append({
                'city':city,'representation':rep,'n_regen_events':len(vals),
                'recovered_sum_mmol_g':vals.sum() if len(vals) else np.nan,
                'recovered_mean_per_event_mmol_g':vals.mean() if len(vals) else np.nan,
                'regen_interval_h':REGEN_INTERVAL_HOURS,'regen_protocol':REGEN_PROTOCOL
            })
    process=pd.DataFrame(process_rows)
    process.to_csv(IRA_RESULTS/'v7_IRA_scheduled_process_scores.csv',index=False)
    display(process.head())


## Comparison to H

Only when a non-RH regeneration schedule has been explicitly selected do we calculate process-ranking fidelity.

Then:

\[
\tau_P=\tau(P_R,P_F)
\]

is compared with the manuscript's

\[
\tau_H=\tau(H_R,H_F).
\]

A schedule sensitivity (24/48/168 h, for example) is a **new process assumption** and must be reported as such. It is not a robustness check on the equilibrium metric.


In [ ]:
if REGEN_INTERVAL_HOURS is not None:
    wide=process.pivot(index='city',columns='representation',values='recovered_mean_per_event_mmol_g').dropna()
    tauP=kendalltau(wide['reduced'],wide['full']).statistic
    rhoP=spearmanr(wide['reduced'],wide['full']).statistic
    print('tau_P =',tauP,'rho_P =',rhoP)

    H=pd.read_csv(H_BASE)
    hrows=[]
    for branch,g in H[H.material==IRA_MATERIAL].groupby('branch'):
        g=g[(g.n_valid>0)&g.H_reduced.notna()&g.H_full.notna()]
        hrows.append({'branch':branch,
                      'tau_H':kendalltau(g.H_reduced,g.H_full).statistic,
                      'rho_H':spearmanr(g.H_reduced,g.H_full).statistic})
    display(pd.DataFrame(hrows))


# Audit statement

v7 deliberately preserves the full hourly RH/T sequence.

The only thresholds in the notebook are:
- experimental support boundaries of the measured \(q(RH,T)\) surface;
- source-reported regeneration conditions stored as metadata.

Neither is used as an ambient adsorption/desorption switching rule.

The remaining unresolved choice is process scheduling. That choice must be supplied explicitly before a productivity-like process quantity is calculated.
